In [1]:
import os
import torch
import pandas as pd
import scanpy as sc
from sklearn import metrics
import multiprocessing as mp

In [2]:
from GraphST import GraphST

In [3]:
# Run device, by default, the package is implemented on 'cpu'. We recommend using GPU.
# device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')

# the location of R, which is necessary for mclust algorithm. Please replace the path below with local R installation path
os.environ['R_HOME'] = r'C:\Program Files\R\R-4.4.2'

In [4]:
# the number of clusters
n_clusters = 7

In [ ]:
dataset = '151673'

In [5]:
import anndata
from GraphST.utils import clustering
import scanpy as sc

n_clusters = 7
slices_name = [ '151674.h5ad','151675.h5ad','151676.h5ad']
folder_path = "E:/SingleCellCluster/Dataset/Visium_standard"
for name in slices_name:
    filepath = folder_path + f"/{name}"
    adata = anndata.read_h5ad(filepath)
    # define model
    model = GraphST.GraphST(adata)
    # train model
    adata = model.train()
    # set radius to specify the number of neighbors considered during refinement
    radius = 50

    tool = 'mclust' # mclust, leiden, and louvain

    # clustering
    if tool == 'mclust':
        clustering(adata, n_clusters, radius=radius, method=tool, refinement=True) # For DLPFC dataset, we use optional refinement step.
    elif tool in ['leiden', 'louvain']:
        clustering(adata, n_clusters, radius=radius, method=tool, start=0.1, end=2.0, increment=0.01, refinement=False) 

    # Bỏ các điểm thiếu ground truth
    adata = adata[~adata.obs['Region'].isna()]
    
    # Chuyển ground truth thành số
    ground_truth_labels = pd.factorize(adata.obs['Region'])[0]
    predicted_labels = adata.obs['domain']
    
    # Tính ARI
    ARI = metrics.adjusted_rand_score(predicted_labels, ground_truth_labels)
    
    # Lưu ARI vào adata.uns (nếu bạn cần)
    adata.uns['ARI'] = ARI
    
    # Lưu đối tượng adata vào tệp .h5ad
    adata.write('Maynard_graphSt_results/adata'+ f"_{name}")

    # In kết quả
    print(f"Slice {name}: ARI = {ARI:.4f}")
    
    

Begin to train ST data...


100%|██████████| 600/600 [11:03<00:00,  1.11s/it]


Optimization finished for ST data!


R[write to console]:                    __           __ 
   ____ ___  _____/ /_  _______/ /_
  / __ `__ \/ ___/ / / / / ___/ __/
 / / / / / / /__/ / /_/ (__  ) /_  
/_/ /_/ /_/\___/_/\__,_/____/\__/   version 6.1.1
Type 'citation("mclust")' for citing this R package in publications.



fitting ...
  |======================================================================| 100%


...

Slice 151675.h5ad: ARI = 0.6186
Begin to train ST data...


100%|██████████| 600/600 [12:35<00:00,  1.26s/it]


Optimization finished for ST data!
fitting ...
  |======================================================================| 100%


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_19020\1271140438.py:37: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  adata.uns['ARI'] = ARI


Slice 151676.h5ad: ARI = 0.5793
